# FinBERT Fine-Tuning for Financial Sentiment Classification

This notebook fine-tunes `ProsusAI/finbert` on the `financial_phrasebank` dataset for 3-class financial sentiment classification. It is designed to run top-to-bottom on a free Google Colab T4 GPU and saves datasets, metrics, plots, and the final model back into the workspace.

## Section 1: Setup & Installs

In [ ]:
# Optional: mount Google Drive in Colab if you want to persist artifacts there.
# from google.colab import drive
# drive.mount('/content/drive')

# Install the required packages explicitly so the notebook is reproducible in a fresh Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers",
    "datasets",
    "evaluate",
    "accelerate",
    "gradio",
    "scikit-learn",
    "seaborn",
    "tqdm",
])

In [ ]:
# Import everything needed for data processing, training, evaluation, and visualization up front.
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from IPython.display import display
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Gradio is installed for optional demo work later, but is not required for training.
import gradio as gr

# Set all requested seeds so runs are as reproducible as possible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use a clean, consistent style for all notebook plots.
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.titlesize": 14, "axes.labelsize": 12})

def resolve_project_root():
    """Find the repository root so the notebook works in Colab and in a local workspace."""
    candidates = [
        os.getcwd(),
        os.path.dirname(os.getcwd()),
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        os.path.abspath(os.path.join(os.getcwd(), "../..")),
        "/content/nlp-a3-financial-sentiment",
        "/content/drive/MyDrive/nlp-a3-financial-sentiment",
    ]
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, "README.md")) and os.path.exists(os.path.join(candidate, "requirements.txt")):
            return candidate
        if os.path.exists(os.path.join(candidate, "data")) and os.path.exists(os.path.join(candidate, "results")):
            return candidate
    return os.getcwd()

PROJECT_ROOT = resolve_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "finbert-financial-sentiment")
FINETUNED_DIR = os.path.join(PROJECT_ROOT, "finbert-finetuned")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINETUNED_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")

## Section 2: Data Loading & Preprocessing

In [ ]:
# Load the Hugging Face dataset and inspect its schema before any preprocessing.
raw_dataset = load_dataset("financial_phrasebank", "sentences_allagree")
print(raw_dataset)
print("\nTrain split features:")
print(raw_dataset["train"].features)

label_id_to_name = {0: "negative", 1: "neutral", 2: "positive"}
label_name_to_id = {v: k for k, v in label_id_to_name.items()}

print("\nClass distribution in the original Hugging Face split:")
print(pd.Series(raw_dataset["train"]["label"]).map(label_id_to_name).value_counts().reindex(label_id_to_name.values()))

In [ ]:
# Convert the dataset to pandas, standardize the column names, and add a string label column for readability.
source_df = raw_dataset["train"].to_pandas()[["sentence", "label"]].rename(columns={"sentence": "text"})
source_df["label_str"] = source_df["label"].map(label_id_to_name)

print(f"DataFrame shape: {source_df.shape}")
print("\nLabel distribution:")
print(source_df["label_str"].value_counts().reindex(label_id_to_name.values()))

# Show three examples per class so the raw text and label mapping are easy to verify.
for label_id, label_name in tqdm(label_id_to_name.items(), desc="Previewing classes"):
    print(f"\nExamples for class: {label_name}")
    display(source_df[source_df["label"] == label_id].head(3))

In [ ]:
# Split the data into train/validation/test sets using stratification to preserve the class balance.
train_df, temp_df = train_test_split(
    source_df,
    test_size=0.30,
    random_state=SEED,
    stratify=source_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{split_name.upper()} shape: {split_df.shape}")
    print(split_df["label_str"].value_counts().reindex(label_id_to_name.values()))

train_csv = os.path.join(DATA_DIR, "train.csv")
val_csv = os.path.join(DATA_DIR, "val.csv")
test_csv = os.path.join(DATA_DIR, "test.csv")

# Save the split CSV files so the preprocessing pipeline leaves behind reusable artifacts.
train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\nSaved CSV files:")
for csv_path in [train_csv, val_csv, test_csv]:
    print(f"{csv_path} -> {os.path.getsize(csv_path) / 1024:.2f} KB")

## Section 3: Tokenization

In [ ]:
# Visualize the label balance across splits to confirm that stratification preserved the distribution.
split_counts = pd.DataFrame(
    {
        "train": train_df["label_str"].value_counts(),
        "val": val_df["label_str"].value_counts(),
        "test": test_df["label_str"].value_counts(),
    }
).reindex(label_id_to_name.values()).fillna(0).astype(int)

split_counts_long = split_counts.reset_index().rename(columns={"index": "label_str"}).melt(
    id_vars="label_str", var_name="split", value_name="count"
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=split_counts_long,
    x="split",
    y="count",
    hue="label_str",
    palette="muted",
    ax=ax,
)
ax.set_title("Financial Sentiment Class Distribution Across Splits")
ax.set_xlabel("Dataset Split")
ax.set_ylabel("Number of Examples")
ax.legend(title="Class")
plt.tight_layout()
plt.show()

In [ ]:
# Load the FinBERT tokenizer, convert pandas frames to Hugging Face Dataset objects, and tokenize in batches.
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

hf_datasets = {
    "train": Dataset.from_pandas(train_df[["text", "label", "label_str"]].reset_index(drop=True), preserve_index=False),
    "val": Dataset.from_pandas(val_df[["text", "label", "label_str"]].reset_index(drop=True), preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label", "label_str"]].reset_index(drop=True), preserve_index=False),
}

tokenized_datasets = {split_name: dataset.map(tokenize, batched=True) for split_name, dataset in hf_datasets.items()}

# Keep a decoded example before changing the dataset format so we can verify tokenization visually.
sample_encoding = tokenized_datasets["train"][0]
print("Decoded training sample:")
print(tokenizer.decode(sample_encoding["input_ids"], skip_special_tokens=True))

# Compute token lengths for the training split and plot the distribution.
train_length_dataset = tokenized_datasets["train"].map(
    lambda batch: {"token_length": [int(sum(mask)) for mask in batch["attention_mask"]]},
    batched=True,
)

format_columns = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in tokenized_datasets["train"].column_names:
    format_columns.insert(2, "token_type_ids")

for split_name in tokenized_datasets:
    tokenized_datasets[split_name].set_format("torch", columns=format_columns)

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(train_length_dataset["token_length"], bins=30, kde=True, color=sns.color_palette("muted")[0], ax=ax)
ax.set_title("Token Length Distribution in the Training Set")
ax.set_xlabel("Token Length")
ax.set_ylabel("Number of Examples")
plt.tight_layout()
plt.show()

print("\nTokenized dataset columns:")
print(tokenized_datasets["train"].column_names)

## Section 4: Model Setup

In [ ]:
# Load FinBERT for sequence classification, then attach the label mapping to the model config.
model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    ignore_mismatched_sizes=True,
)

model.config.id2label = label_id_to_name
model.config.label2id = label_name_to_id

total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Model config label mapping:")
print(model.config.id2label)

## Section 5: Fine-tuning

In [ ]:
# Define metrics, training hyperparameters, and the Trainer. The evaluation strategy is set to epoch-level checks to match the requested workflow while remaining compatible with the installed transformers version.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    weighted_f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": macro_f1, "weighted_f1": weighted_f1}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

start_time = time.time()
train_output = trainer.train()
training_time_seconds = time.time() - start_time

print(f"\nTraining finished in {training_time_seconds / 60:.2f} minutes")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best validation metric: {trainer.state.best_metric}")

# Plot the training loss and evaluation F1 traces from the trainer logs.
log_history = trainer.state.log_history
train_steps = []
train_losses = []
eval_epochs = []
eval_f1_scores = []

for entry in log_history:
    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_f1" in entry and "epoch" in entry:
        eval_epochs.append(entry["epoch"])
        eval_f1_scores.append(entry["eval_f1"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_steps, train_losses, marker="o", color=sns.color_palette("muted")[0], label="Training Loss")
axes[0].set_title("Training Loss vs Steps")
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(eval_epochs, eval_f1_scores, marker="o", color=sns.color_palette("muted")[1], label="Eval F1")
axes[1].set_title("Validation F1 vs Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 6: Evaluation on Test Set

In [ ]:
# Evaluate the fine-tuned model on the untouched test set and compute the required metrics.
test_predictions = trainer.predict(tokenized_datasets["test"])
test_logits = test_predictions.predictions
test_true = test_predictions.label_ids
test_pred = np.argmax(test_logits, axis=-1)

test_accuracy = accuracy_metric.compute(predictions=test_pred, references=test_true)["accuracy"]
test_macro_f1 = f1_metric.compute(predictions=test_pred, references=test_true, average="macro")["f1"]
test_weighted_f1 = f1_metric.compute(predictions=test_pred, references=test_true, average="weighted")["f1"]

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro F1: {test_macro_f1:.4f}")
print(f"Test weighted F1: {test_weighted_f1:.4f}")

report_dict = classification_report(
    test_true,
    test_pred,
    target_names=[label_id_to_name[i] for i in range(3)],
    output_dict=True,
)

print("\nClassification report:")
print(classification_report(
    test_true,
    test_pred,
    target_names=[label_id_to_name[i] for i in range(3)],
    digits=4,
))

# Plot the confusion matrix twice: once with raw counts and once with normalized percentages.
cm_raw = confusion_matrix(test_true, test_pred, labels=[0, 1, 2])
cm_norm = confusion_matrix(test_true, test_pred, labels=[0, 1, 2], normalize="true") * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(
    cm_raw,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[label_id_to_name[i] for i in range(3)],
    yticklabels=[label_id_to_name[i] for i in range(3)],
    ax=axes[0],
    cbar=False,
)
axes[0].set_title("Test Confusion Matrix - Raw Counts")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Actual Label")

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=[label_id_to_name[i] for i in range(3)],
    yticklabels=[label_id_to_name[i] for i in range(3)],
    ax=axes[1],
    cbar=True,
)
axes[1].set_title("Test Confusion Matrix - Normalized (%)")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

# Build the requested summary table and leave placeholders for the comparison models.
results_table = pd.DataFrame([
    {
        "Model": "FinBERT (fine-tuned)",
        "Accuracy": test_accuracy,
        "Macro F1": test_macro_f1,
        "F1-negative": report_dict["negative"]["f1-score"],
        "F1-neutral": report_dict["neutral"]["f1-score"],
        "F1-positive": report_dict["positive"]["f1-score"],
    },
    {"Model": "VADER", "Accuracy": None, "Macro F1": None, "F1-negative": None, "F1-neutral": None, "F1-positive": None},
    {"Model": "TextBlob", "Accuracy": None, "Macro F1": None, "F1-negative": None, "F1-neutral": None, "F1-positive": None},
    {"Model": "DistilBERT", "Accuracy": None, "Macro F1": None, "F1-negative": None, "F1-neutral": None, "F1-positive": None},
])

results_csv_path = os.path.join(RESULTS_DIR, "results_table.csv")
results_table.to_csv(results_csv_path, index=False)
print("\nResults summary table:")
display(results_table)
print(f"Saved results table to: {results_csv_path}")

## Section 7: Error Analysis

In [ ]:
# Attach predictions to the test DataFrame so we can inspect mistakes in context.
test_results_df = test_df.reset_index(drop=True).copy()
test_results_df["predicted"] = test_pred
test_results_df["predicted_str"] = test_results_df["predicted"].map(label_id_to_name)

misclassified_df = test_results_df[test_results_df["label"] != test_results_df["predicted"]].copy()

# Show up to five misclassified examples per true class using tqdm so the manual loop is explicit and visible.
for label_id, label_name in tqdm(label_id_to_name.items(), desc="Collecting misclassifications"):
    subset = misclassified_df[misclassified_df["label"] == label_id].head(5)
    print(f"\nTop misclassified examples where the true label is {label_name}:")
    if subset.empty:
        print("No misclassifications found for this class.")
    else:
        display(subset[["text", "label_str", "predicted_str"]])

# Identify the most confused class pairs by sorting the off-diagonal cells of the confusion matrix.
cm_df = pd.DataFrame(
    cm_raw,
    index=[label_id_to_name[i] for i in range(3)],
    columns=[label_id_to_name[i] for i in range(3)],
)
off_diagonal = cm_df.where(~np.eye(cm_df.shape[0], dtype=bool)).stack().sort_values(ascending=False)

print("\nMost confused class pairs:")
for (true_label, predicted_label), count in off_diagonal.head(5).items():
    print(f"{true_label} -> {predicted_label}: {int(count)}")

## Section 8: Inference on Custom Text

In [ ]:
# Build a reusable inference helper that returns the predicted label and per-class confidence scores.
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1).squeeze(0).detach().cpu().numpy()

    confidence_scores = {label_id_to_name[i]: float(probabilities[i]) for i in range(3)}
    predicted_label = max(confidence_scores, key=confidence_scores.get)
    return predicted_label, confidence_scores

custom_sentences = [
    "The company reported record revenue growth and raised its full-year guidance.",
    "Management said results were in line with expectations and maintained the dividend.",
    "The firm warned that lower demand could pressure margins in the next quarter.",
    "Shares fell after the credit rating agency downgraded the issuer to junk status.",
    "The acquisition is expected to create synergies and improve earnings over time.",
]

prediction_rows = []
for sentence in custom_sentences:
    predicted_label, confidence_scores = predict_sentiment(sentence)
    prediction_rows.append({
        "Sentence": sentence,
        "Predicted": predicted_label,
        "Confidence": max(confidence_scores.values()),
        "negative": confidence_scores["negative"],
        "neutral": confidence_scores["neutral"],
        "positive": confidence_scores["positive"],
    })

custom_predictions_df = pd.DataFrame(prediction_rows)
print("Custom sentiment predictions:")
display(custom_predictions_df)

## Section 9: Save Model

In [ ]:
# Save the fine-tuned model and tokenizer so they can be reused without retraining.
model.save_pretrained(FINETUNED_DIR)
tokenizer.save_pretrained(FINETUNED_DIR)

print(f"Saved fine-tuned artifacts to: {FINETUNED_DIR}")
print("Directory contents:")
for item in sorted(os.listdir(FINETUNED_DIR)):
    print(f" - {item}")